# Connect Four endgames beyond ten empty cells

This notebook continues the $7\times6$ endgame scaling study of *Certify, Then Repair*
past the ten-empty-cell arm reported in the paper. Nothing here is a new method: it is the
same `src/c4_endgame.py`, the same 40-root suites and the same training protocol. The only
change is scheduling — the work is spread over several processes instead of queueing on
two laptop cores.

**What the two-core laptop measured.** One 10-empty-cell repair arm (seed 1) certified in
2,502 s. The same protocol *with both matched controls* (seed 0) took 41,192 s, of which
34,700 s, or 84%, is controls. Controls are matched to the repair arm's per-round history
budgets but almost never satisfy the stopping rule, so every one of their rounds burns all
200 epochs. The expensive part is therefore also the part that parallelises for free: two
independent control arms per seed.

**Why splitting costs nothing scientifically.** `--controls-from` re-runs the matched arms
of an archived repair arm from the per-round budgets recorded in its JSON, starting from
the same policy (and `--base-fit` re-derives that starting policy when an older archive
never saved one). `tests/test_c4_endgame.py::test_controls_can_run_from_a_saved_repair_and_match_inline`
asserts that the split arms reproduce the inline arms' per-round certificate logs exactly.
Splitting is a scheduling decision, not a change of experiment.

**GPU: no, and that is not a shortage of money.** The certificate is a pure-Python
backward-induction recursion and the policy is a 57k-parameter GRU; a T4 accelerates
neither, and CUDA start-up only adds latency. Use Runtime $\to$ Change runtime type $\to$
**CPU**. What a higher-tier runtime buys is *cores*, and section 3 spends them: the queue
keeps one arm per usable core.

**Free-tier caveat.** A session is capped at roughly 12 h and a runtime restart kills every
child process. The queue cell is resumable: it skips arms whose JSON already exists,
attaches to arms still running, and mirrors finished JSONs and checkpoints to Drive on each
pass. Close the tab, come back tomorrow, run the same cells again.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
PERSIST = pathlib.Path('/content/drive/MyDrive/c4_scaling')
PERSIST.mkdir(parents=True, exist_ok=True)
print('outputs will be copied to', PERSIST)

In [ ]:
!git clone --depth 1 https://github.com/starlyn2010/fragility-paper.git
%cd /content/fragility-paper
!python -c "import torch, platform; print(platform.python_version(), torch.__version__)"

## 1. How much machine do we actually have

`os.cpu_count()` reports the *host*, not the slice you are allowed to use, so read the
cgroup quota instead. Everything downstream keys off `MAX_PAR`.

In [ ]:
import os

def quota_cores():
    try:                                       # cgroup v2
        quota, period = open('/sys/fs/cgroup/cpu.max').read().split()
        if quota != 'max':
            return max(1, int(float(quota) / int(period)))
    except Exception:
        pass
    try:                                       # cgroup v1
        quota = int(open('/sys/fs/cgroup/cpu/cpu.cfs_quota_us').read())
        if quota > 0:
            return max(1, quota // int(open('/sys/fs/cgroup/cpu/cpu.cfs_period_us').read()))
    except Exception:
        pass
    return os.cpu_count() or 2

def avail_gb():
    for line in open('/proc/meminfo'):
        if line.startswith('MemAvailable'):
            return int(line.split()[1]) / 2 ** 20

CORES, RAM = quota_cores(), avail_gb()
# An arm is one single-threaded recursion plus a small GRU refit; budget ~2.5 GB per
# concurrent process and never oversubscribe the CPUs.
MAX_PAR = max(1, min(CORES, int(RAM // 2.5)))
THREADS = max(1, CORES // MAX_PAR)
print(f'usable cores {CORES}, available RAM {RAM:.1f} GB')
print(f'=> {MAX_PAR} arms at a time, --threads {THREADS} each')

## 2. Cost probe before committing to a suite

Times one exact oracle solve per root at the chosen depth. Multiply by 40 roots for the
cost of one certificate pass; a repair arm with $R$ rounds and its two control arms do
roughly $R+2$, $R$ and $R$ passes, so this probe plus the laptop timings above is enough to
budget a run. A free CPU instance usually reports `usable cores 2`, which still halves each
wave.

In [ ]:
import time
import src.c4_game as g
from src.c4_endgame import random_games

g.configure(6, 7, 4)

def probe(empties, n=6, seed=7):
    # Sample with the same code the run uses, then time a *fresh* Oracle so the numbers
    # reflect real solving rather than a warm memo.
    sampler = g.Oracle()
    t0 = time.time()
    roots, games = random_games(sampler, empties, n, seed, 0)
    t_sample = time.time() - t0
    if len(roots) < n:
        print(f'{empties:2d} empty cells: only {len(roots)} draw roots found '
              f'in {games} games ({t_sample:.1f} s)')
        return
    oracle = g.Oracle()
    times = []
    for r in roots:
        b = tuple(r['board'])
        start = time.time()
        oracle.value(b)
        oracle.optimal(b)
        times.append(time.time() - start)
    warm = sum(times[1:]) / max(len(times) - 1, 1)
    print(f'{empties:2d} empty cells: sampled {n} roots from {games} games in '
          f'{t_sample:5.1f} s | cold first root {times[0]:6.3f} s, '
          f'{warm:6.3f} s per root once the memo is warm')

for e in (10, 12, 14):
    probe(e)
print('the two-core laptop measured ~0.10 s per root at 10 empty cells, '
      'so the ratios above are the extra depth cost')

## 3. The queue

`build(12)` queues every arm of a depth and how they depend on each other:

* **3 repair arms** (seeds 0, 1, 2) — the CGR loop, run with `--no-controls` so all three
  start immediately and each one frees a slot early.
* **6 control arms** — two per seed (`all`, `tree`), each waiting for that seed's repair
  JSON and base checkpoint. This is the 84%-of-the-time work and the part that gains most
  from concurrency.
* **3 verification arms** — the independent re-derivation, waiting for the final checkpoint.

The protocol flags are the ten-cell protocol: 40 draw-valued roots, hidden 128, each added
history repeated 60 times, a refit stops only when replay **and** aggregate are exactly
optimal, up to 24 rounds. `--epochs`, `--agg-repeat`, `--stop` and `--max-iters` are part
of what makes a deeper arm comparable to the published one, so they are not exposed as
knobs here.

In [ ]:
import json, os, pathlib, shutil, subprocess

SEEDS = (0, 1, 2)
CONTROL_SEEDS = (0,)            # the published arms archive controls for one seed per depth
CONTROL_SEED = 1             # the optimizer seed those controls use

def build(empties, seeds=SEEDS):
    """Queue the repair, matched-control and verification arms for one depth."""
    E = empties
    roots_file = f'data/c4_endgame_roots_{E}_40_s0.json'
    common = ['--rows', '6', '--cols', '7', '--win', '4', '--empties', str(E),
              '--roots', '40', '--root-value', '0', '--branch', '3',
              '--hidden', '128', '--emb', '16', '--epochs', '200', '--lr', '2e-3',
              '--agg-repeat', '60', '--stop', 'both', '--max-iters', '24',
              '--roots-file', roots_file, '--threads', str(THREADS)]

    def repair(s):
        out = f'data/c4_end{E}_s{s}.json'
        return dict(name=f'repair s{s}', out=out, needs=[],
                    cmd=['python', 'src/c4_endgame.py'] + common +
                         ['--seed', str(s), '--no-controls',
                          '--ckpt-out', f'runs/c4_end{E}_s{s}', '--out', out])

    def control(s, mode):
        src = f'data/c4_end{E}_s{s}.json'
        out = f'data/c4_end{E}_s{s}_{mode}.json'
        # the per-round budgets and the starting policy both come from the archived arm
        return dict(name=f'{mode} control s{s}', out=out,
                    needs=[src, f'runs/c4_end{E}_s{s}/c4_gru_base.pt'],
                    cmd=['python', 'src/c4_endgame.py'] + common +
                         ['--seed', str(s), '--controls-from', src,
                          '--control-mode', mode, '--control-seeds', str(CONTROL_SEED),
                          '--control-games', '3', '--out', out])

    def verify(s):
        src = f'data/c4_end{E}_s{s}.json'
        out = f'data/c4_end{E}_verify_s{s}.json'
        ckpt = f'runs/c4_end{E}_s{s}/c4_gru.pt'
        return dict(name=f'verify s{s}', out=out, needs=[src, ckpt],
                    cmd=['python', 'src/c4_endgame_verify.py', '--ckpt', ckpt,
                         '--roots', roots_file, '--hidden', '128', '--emb', '16',
                         '--orders', '120', '--games', '40', '--out', out])

    jobs = ([repair(s) for s in seeds]
            + [control(s, m) for s in CONTROL_SEEDS for m in ('all', 'tree')]
            + [verify(s) for s in seeds])
    for j in jobs:
        j['log'] = 'runs/_colab_' + j['out'].split('/')[-1][:-5] + '.log'
    return E, roots_file, jobs

E, ROOTS_FILE, JOBS = build(12)
print(f'{len(JOBS)} arms queued for {E} empty cells ({MAX_PAR} at a time): repair arms '
      f'first, then the controls and verifications they unblock')
print('controls cover seeds', CONTROL_SEEDS, '- widen the tuple for the other seeds, but a '
      'control arm costs roughly what its repair arm cost, per round')
print('for 14 empty cells, re-run this cell as `E, ROOTS_FILE, JOBS = build(14)`')

## 4. Sample the suite once, up front

Every arm must see the *identical* 40 roots, and sampling is oracle work: letting each of
the nine processes draw its own suite would repeat that work nine times and, worse, compare
seeds on different positions. This cell writes `data/c4_endgame_roots_{E}_40_s0.json` with
the same `random_games(oracle, E, 40, 0, 0)` call a seed-0 run makes, so the file is
interchangeable with a laptop one. The larger game cap only prevents giving up early: the
first 40 accepted roots do not depend on it.

In [ ]:
import json, pathlib, time
import src.c4_game as g
from src.c4_endgame import random_games

g.configure(6, 7, 4)
t0 = time.time()
path = pathlib.Path(ROOTS_FILE)
if path.exists():
    print(f'reusing {len(json.loads(path.read_text()))} roots from {path}')
else:
    roots, games = random_games(g.Oracle(), E, 40, 0, 0, max_games=500000)
    if len(roots) < 40:
        raise SystemExit(f'only {len(roots)} draw-valued roots in {games} games; '
                         'raise max_games rather than shrinking the suite')
    path.write_text(json.dumps(roots))
    print(f'sampled 40 roots from {games} games in {time.time() - t0:.0f} s -> {path}')

## 5. Launch

`tick()` is idempotent: run it once, then again whenever you come back to the tab. It
starts whatever is unblocked while keeping at most `MAX_PAR` processes alive, then mirrors
the JSONs and checkpoints to Drive.

In [ ]:
def pids(needle):
    """Pids whose command line contains `needle`; callers pass 'out <path>',
    without the leading dashes, because pgrep would read --out as its own option."""
    r = subprocess.run(['pgrep', '-f', needle], capture_output=True, text=True)
    return [int(x) for x in r.stdout.split()]

def sync():
    for f in sorted(pathlib.Path('data').glob(f'c4_end{E}_*.json')):
        dst = PERSIST / f.name
        if not dst.exists() or dst.stat().st_size != f.stat().st_size:
            shutil.copy(f, dst)
    for f in sorted(pathlib.Path('data').glob(f'c4_endgame_roots_{E}_*.json')):
        shutil.copy(f, PERSIST / f.name)
    keep = PERSIST / f'ckpt_end{E}'
    keep.mkdir(parents=True, exist_ok=True)
    for f in sorted(pathlib.Path('runs').glob(f'c4_end{E}_s*/c4_gru*.pt')):
        dst = keep / f'{f.parent.name}_{f.name}'
        if not dst.exists() or dst.stat().st_size != f.stat().st_size:
            shutil.copy(f, dst)

def tick():
    live = sum(1 for j in JOBS if pids('out ' + j['out']))
    for j in JOBS:
        if os.path.exists(j['out']) or pids('out ' + j['out']):
            continue
        if not all(os.path.exists(p) for p in j['needs']):
            continue
        if live >= MAX_PAR:
            break
        fh = open(j['log'], 'w')
        subprocess.Popen(j['cmd'], stdout=fh, stderr=subprocess.STDOUT,
                         stdin=subprocess.DEVNULL)
        live += 1
        print('started', j['name'], '->', j['log'], flush=True)
    sync()
    left = [j['name'] for j in JOBS if not os.path.exists(j['out'])]
    print(f'{len(JOBS) - len(left)}/{len(JOBS)} JSONs on disk | {live} running | '
          f'{len(left)} to go')

tick()

### Monitor

Re-run this every few minutes. For a repair arm the last log line is the current round's
worst-case draw rate and violation count, so `worst` falling towards 1.0 is the story.

In [ ]:
for j in JOBS:
    if os.path.exists(j['out']):
        print(f'  done  {j["name"]}')
    elif pids('out ' + j['out']):
        tail = ''
        if os.path.exists(j['log']):
            lines = [l for l in open(j['log']).read().splitlines() if l.strip()]
            tail = lines[-1] if lines else ''
        print(f'  RUN   {j["name"]} | {tail[:104]}')
    elif not all(os.path.exists(p) for p in j['needs']):
        print(f'  wait  {j["name"]}')
    else:
        print(f'  ready {j["name"]} (a slot is free - run tick() again)')

if not [j for j in JOBS if not os.path.exists(j['out'])]:
    print('queue empty')

## 6. Read the result

A `worst_draw_rate` of 1.0 means all 40 roots are forced draws however the optimal teacher
breaks its ties, which is the guarantee the paper claims. The controls are matched by
construction: their per-round added-history budgets are the repair arm's, so the comparison
is histories-for-histories rather than epochs-for-epochs.

In [ ]:
for s in SEEDS:
    f = f'data/c4_end{E}_s{s}.json'
    if not os.path.exists(f):
        continue
    d = json.load(open(f))
    fin = d['repair']['final']
    print(f"seed {s}: W {fin['worst_draw_rate']:.2f} B {fin['best_draw_rate']:.2f} "
          f"D {fin['random_draw_rate_exact']:.3f} | rounds {len(d['repair']['log']) - 1} "
          f"| added {len(d['repair']['aggregate'])} | repair {d['repair']['elapsed_s']:.0f}s "
          f"| file {d['elapsed_s']:.0f}s")
    for mode in ('all', 'tree'):
        cf = f'data/c4_end{E}_s{s}_{mode}.json'
        if os.path.exists(cf):
            c = json.load(open(cf))
            arm = c['controls'][0]
            print(f"    control {mode:4s}: W {arm['final']['worst_draw_rate']:.2f} "
                  f"B {arm['final']['best_draw_rate']:.2f} "
                  f"D {arm['final']['random_draw_rate_exact']:.3f} "
                  f"| rounds {len(arm['log']) - 1} | added {arm['final']['added']} "
                  f"| {c['elapsed_s']:.0f}s")
    vf = f'data/c4_end{E}_verify_s{s}.json'
    if os.path.exists(vf):
        print('    verification:', json.load(open(vf)).get('summary'))

print()
print('copied to', PERSIST, '-', sorted(p.name for p in PERSIST.glob(f'c4_end{E}_*.json')))

## 7. Optional: give the laptop's ten-cell seed 1 its controls

Seed 1 was certified on the laptop before base checkpoints were archived, so its JSON
carries no matched controls. `--base-fit` re-derives the exact starting policy from the
archived replay set and refuses to continue unless it reproduces the archived base
certificate; the test quoted in the header pins that behaviour. Each control round costs
about a thousand seconds at ten empty cells (the seed-0 file spent 34,700 s on 34 control
rounds), so this is roughly four hours of one core per arm - run the two of them
concurrently and they are one four-hour block.

In [ ]:
# %%bash
# cd /content/fragility-paper
# BASE=(--rows 6 --cols 7 --win 4 --empties 10 --roots 40 --root-value 0 --branch 3
#       --hidden 128 --emb 16 --epochs 200 --lr 2e-3 --agg-repeat 60 --stop both
#       --max-iters 24 --seed 1 --roots-file data/c4_endgame_roots_10_40_s0.json
#       --ckpt-out runs/c4_end10_s1 --controls-from data/c4_end10_s1.json)
# # step 1: re-derive the starting policy and check it against the archived certificate
# python src/c4_endgame.py "${BASE[@]}" --threads 1 --base-fit --no-controls
# # step 2: the two matched controls, concurrently
# for mode in all tree; do
#   python src/c4_endgame.py "${BASE[@]}" --threads 2 --base-ckpt runs/c4_end10_s1/c4_gru_base.pt \
#     --control-mode $mode --control-seeds 1 --control-games 3 \
#     --out data/c4_end10_s1_$mode.json &
# done
# wait

## 8. Fourteen empty cells

Run `E, ROOTS_FILE, JOBS = build(14)` in the queue cell, then the sampling cell, then
`tick()`. The 12-cell JSONs already on disk are not touched, because every filename is
keyed on the depth. Budget it from the section 2 probe: at 14 cells one root of oracle work
is several times the 10-cell laptop number, and each arm pays a pass per round, so expect
the 14-cell queue to need more than one session. Drive already holds every finished JSON and
checkpoint, so nothing is lost between them. A 14-empty suite can also need the wider
sampling cap; section 4 raises it to 500,000 games and stops rather than quietly returning a
shorter suite.